# ⚡ **<font color="steelblue">EC4. Gestión de Riesgos en  "Eco-Flow Renewables"</font>**

**Bloque:** Variables aleatorias y simulación

**Fecha última edición**: 08/09/2026

**Licencia**: <small><a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a><br /></small>  

No olvides hacer una copia de este cuaderno (`Archivo > Guardar una copia en Drive`) antes de empezar a trabajar.

---
---

In [ ]:
#%%capture
# @title ⚠️ Cargar configuración del cuaderno
# Cargamos módulos de análisis numérico
import numpy as np          # importamos numpy como np
import pandas as pd         # importamos pandas como pd
import math                 # importamos módulo para cáculos matemáticos
import random
import inspect
from scipy import stats
import matplotlib.colors as mcolors # Importamos matplotlib.colors

# Cargamos módulos de análisis gráficos
from plotnine import *      # importamos módulo para gráficos con ggplot
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
%config InlineBackend.figure_format = 'retina'

#===============================================
# Cargar funciones bloque1
from urllib.request import urlretrieve
import re # for text manipulation


#url = 'https://raw.githubusercontent.com/asunmayoral/umh1477/refs/heads/main/bloque1_sps.py'
url = 'https://raw.githubusercontent.com/UMH1477/python/refs/heads/main/bloque1_sps.py'
urlretrieve(url, "bloque1_sps.py")
import bloque1_sps
from bloque1_sps import *

# visualización de las funciones precargadas
functions = []
for name, obj in inspect.getmembers(bloque1_sps):
    if inspect.isfunction(obj) and obj.__module__ == bloque1_sps.__name__:
        functions.append(name)

print("\nFunciones precargadas en bloque1_sps.py:\n")
for func_name in functions:
    print(f"- {func_name}")


Funciones precargadas en bloque1_sps.py:

- MC_estim
- best_fit_continuous
- best_fit_continuous_v2
- best_fit_discrete
- calculate_chi2_robust
- cmtd_matrix_n
- distr_discreta
- gof_continuous
- graficar_discreta
- mat_ocupacion_proceso
- obtener_distribucion_ganadora
- simula_discreta


---
---

# **<font color="brown">1. Introducción</font>**

En los últimos años, cada vez más empresas industriales están apostando por generar su propia energía en lugar de depender por completo de la red eléctrica general. **"Eco-Flow Renewables"** es una de ellas: gestiona una **micro-red eléctrica**, una instalación local que combina tres fuentes de energía renovable — **Solar**, **Eólica** e **Hidráulica** — para abastecer directamente a un conjunto de clientes industriales cercanos. La idea de negocio es atractiva: energía más barata y más limpia, sin los costes de transporte ni las emisiones asociadas a la red convencional.

El problema es que la naturaleza no firma contratos. Un panel solar no genera nada de noche ni en un día nublado; un aerogenerador no gira si no hay viento; una turbina hidráulica depende del caudal de un río que a su vez depende de la lluvia caída semanas atrás. Estas tres fuentes, además, no fallan siempre a la vez: un día nublado y sin sol puede ser, precisamente, un día de mucho viento. Por eso "Eco-Flow Renewables" combina las tres fuentes: para que se compensen entre sí. Pero esa compensación nunca es perfecta, y **el estado climático del día determina, en última instancia, cuánta energía es capaz de producir la micro-red**. A este fenómeno lo llamamos **intermitencia**, y es el reto central de cualquier negocio basado en renovables.

Aquí es donde aparece el riesgo financiero. Los clientes industriales de "Eco-Flow Renewables" consumen una cantidad de energía (**la demanda**) que no espera a que salga el sol: sus máquinas necesitan electricidad todos los días, haga el tiempo que haga. Cuando la generación de la micro-red no alcanza a cubrir esa demanda, la empresa está contractualmente obligada a comprar la energía que falta en el **mercado spot** —el mercado donde la electricidad se compra y se vende a muy corto plazo, a veces hora a hora— a un precio bastante más alto que el que "Eco-Flow Renewables" cobra cuando es ella quien tiene excedente y lo vende a la red. Esta asimetría de precios (comprar caro en los días malos, vender barato en los días buenos) es lo que puede convertir un negocio aparentemente rentable en una fuente constante de pérdidas si no se gestiona con cuidado.

La dirección financiera de la empresa no puede permitirse el lujo de esperar diez años para ver cómo se comporta el negocio "en la práctica" y descubrir entonces si es viable o no. Necesita respuestas ahora, y las necesita en términos de probabilidades y de riesgo, no de certezas: no existe un único "día típico", sino un abanico enorme de días posibles según el clima que toque.

Te incorporas al equipo de "Eco-Flow Renewables" como **analista cuantitativo de riesgos**. Tu misión es construir ese simulador y usarlo para responder, con números y con intervalos de confianza, las preguntas que la dirección financiera lleva meses haciéndose: ¿qué beneficio (o pérdida) debemos esperar en un día cualquiera? ¿con qué frecuencia nos vamos a quedar cortos de energía? ¿y si tenemos verdadera mala suerte —una racha de días grises seguidos— cuánto podemos llegar a perder?

Para llegar a esas respuestas de forma progresiva, trabajarás en dos situaciones:

* **Situación Básica.** Empezamos por lo más simple: modelamos el balance energético y el beneficio de **un único día**, tratado de forma aislada, sin memoria de lo ocurrido el día anterior. Aquí construirás el simulador completo —aprendiendo a generar el escenario climático y, condicionada a él, la generación y la demanda— y extraerás las primeras conclusiones sobre el riesgo real del negocio.

* **Situación Avanzada.** Una vez entendido el caso de un solo día, la empresa se plantea una mejora concreta: instalar una **batería de almacenamiento** de 20 MWh que guarde el excedente de los días buenos para poder usarlo en los días malos, en lugar de venderlo y comprarlo de vuelta a un precio mucho peor. Esto cambia las reglas del juego: el estado de la batería (cuánta energía tiene guardada) **se arrastra de un día al siguiente**, así que ya no podrás simular cada día por separado — tendrás que simular la micro-red **día a día, a lo largo de un año completo**, dejando que la batería "recuerde" lo ocurrido antes, para poder responder a la pregunta que de verdad le interesa a la empresa: ¿compensa económicamente esa inversión?

## <font color="brown">**Tu encargo**</font>

No se te pide solo código: se te pide una **recomendación fundamentada**. Al terminar este estudio de caso deberás entregar dos productos, como haría cualquier asesor de ciencia de datos en un proyecto real de consultoría:

* Un **informe técnico** (documento escrito) que recoja el planteamiento del problema, el modelo utilizado, los resultados de la simulación (con sus intervalos de confianza — nunca un único número suelto) y tu interpretación de negocio de cada resultado.
* Una **presentación ejecutiva** pensada para el comité de dirección de "Eco-Flow Renewables": personas que no van a leer tu código ni tus fórmulas, y que necesitan entender, en el menor tiempo posible, cuál es la situación de riesgo actual de la empresa y qué recomiendas hacer al respecto.

Ve tomando nota, a medida que resuelvas cada tarea, de qué resultado usarías en el informe y cuál mostrarías en la presentación: no es necesariamente el mismo nivel de detalle.

# **<font color="brown">2. Balance energético diario</font>**

El balance energético diario establecido actualmente por la empresa viene descrito a continuación.

## **2.1. Definición del proceso**

El primer paso es considerar el **escenario climático ($T$)** del día. Este indicador determina el comportamiento de todas las fuentes de generación y de la demanda:

| Escenario ($T$) | Descripción | Prob. ($p_j$) |
| :--- | :--- | :--- |
| **$T=1$** | Estival (sol intenso, calma de viento) | $0.40$ |
| **$T=2$** | Frontal (muy nublado, vientos fuertes) | $0.30$ |
| **$T=3$** | Monzónico (lluvia extrema, hidro alta) | $0.20$ |
| **$T=4$** | Anticiclónico (día gris, sin viento ni sol) | $0.10$ |

Dependiendo del escenario $T$, las tres fuentes de generación (en MWh) y la demanda de los clientes industriales ($D$, en MWh) se simulan de forma independiente entre sí **dado $T$**:

| Escenario ($T$) | Solar $G_s$ (Triangular)______ | Eólica $G_w$ (Weibull)__________ | Hidro $G_h$ (Lognormal)__________ | Demanda $D$ (Normal)______ |
| :--- | :--- | :--- | :--- | :--- |
| **1. Estival** | $Tri(80,120,150)$ | $Weib(k=1.5,\lambda=5)$ | $LogN(\mu=2,\sigma=0.2)$ | $N(135,10)$ |
| **2. Frontal** | $Tri(0,15,35)$ | $Weib(k=3.0,\lambda=80)$ | $LogN(\mu=2.5,\sigma=0.3)$ | $N(115,15)$ |
| **3. Monzónico** | $Tri(0,10,15)$ | $Weib(k=2.0,\lambda=40)$ | $LogN(\mu=4,\sigma=0.5)$ | $N(125,15)$ |
| **4. Anticiclónico** | $Tri(5,15,20)$ | $Weib(k=1.2,\lambda=10)$ | $LogN(\mu=2,\sigma=0.1)$ | $N(145,10)$ |

> **Notas.** En la Triangular, los tres parámetros son (mínimo, moda, máximo). En la Weibull, $k$ es el parámetro de forma y $\lambda$ el de escala. En la Lognormal, $\mu$ y $\sigma$ son la media y la desviación típica del **logaritmo** de la variable.

Con las características de generación y demanda del escenario del día, el beneficio neto diario ($Y$) se calcula siguiendo este flujo lógico:

1. **Generación total:** $G_{total} = G_s + G_w + G_h$.
2. **Balance neto:** $B = G_{total} - D$.
3. **Regla de mercado:**
   * Si $B\ge 0$ (superávit), la energía sobrante se vende a la red a un precio de exportación $P_{exp}=40$ €/MWh.
   * Si $B<0$ (déficit), hay que comprar la energía que falta en el mercado spot a un precio de penalización $P_{pen}=80$ €/MWh.
4. **Costes operativos** (independientes del balance): un coste fijo diario $C_{fijo}=1.000$€ diarios, y un coste variable de desgaste de equipos $C_{var}=2.0\cdot G_{total}$ (€ por cada MWh generado, se produzca o no superávit).

$$Y = \begin{cases} B\cdot P_{exp} - C_{fijo} - C_{var} & \text{si } B\ge 0 \\ B\cdot P_{pen} - C_{fijo} - C_{var} & \text{si } B< 0 \end{cases}$$

## **2.2. Simulación del proceso**

En este punto describimos el pseudocódigo del algoritmmo y se proporciona el código necesario para la simulación de un conjunto específico de días de funcionamiento.

### **Pseudocódigo simulación del proceso**


$T$, $G_s$, $G_w$, $G_h$ y $D$ se simulan mediante el **método de composición**: primero el escenario climático, después las variables condicionadas a él.

1. Simular $t_i\sim T$ (escenarios 1 a 4, con probabilidades $0.40,\ 0.30,\ 0.20,\ 0.10$).

2. Simular, condicionado a $t_i$ (independientes entre sí dado $t_i$):
    * $g_{s,i}\sim Tri$,
    * $g_{w,i}\sim Weib$,
    * $g_{h,i}\sim LogN$,
    * $d_i\sim N$

3. Calcular:
    * $G_{total,i}=g_{s,i}+g_{w,i}+g_{h,i}$
    * $B_i=G_{total,i}-d_i$.

4. Calcular el beneficio neto $y_i$ aplicando la regla de mercado según el signo de $B_i$.

5. Repetir los pasos 1-4 $nsim$ veces para obtener la muestra $\{y_i\}$.


### **Código simulación del proceso**

A contnuación se muestra el código necesrio apra simular el comportamiento de un día leatoria siguiendo el pseudocódigo anterior. El cósigo se estrutura en tre bloques:
* Bloque 1: parámetros fijos del modelo.
* Bloque 2: generación de escenarios (`_generar_generacion_demanda()`).
* Bloque 3: simulador de un día de funcionamiento (`simulador()`).

In [ ]:
#@title **Parámetros del modelo**

# --- Escenarios climáticos y su distribución ---
ESCENARIOS = [1, 2, 3, 4]              # 1=Estival, 2=Frontal, 3=Monzonico, 4=Anticiclonico
PROBS_ESCENARIO = [0.40, 0.30, 0.20, 0.10]

# --- Distribuciones condicionadas Gs|T, Gw|T, Gh|T, D|T ---
# Gs ~ Triangular(min, moda, max); Gw ~ Weibull(k, escala); Gh ~ Lognormal(mu, sigma); D ~ Normal(mu, sigma)
PARAMS_ESCENARIO = {
    1: dict(tri_s=(80, 120, 150), weib_w=(1.5, 5),  ln_h=(2.0, 0.2), norm_d=(135, 10)),  # Estival
    2: dict(tri_s=(0, 15, 35),    weib_w=(3.0, 80), ln_h=(2.5, 0.3), norm_d=(115, 15)),  # Frontal
    3: dict(tri_s=(0, 10, 15),     weib_w=(2.0, 40), ln_h=(4.0, 0.5), norm_d=(125, 15)),  # Monzonico
    4: dict(tri_s=(5, 15, 20),    weib_w=(1.2, 10), ln_h=(2.0, 0.1), norm_d=(145, 10)),  # Anticiclonico
}

# --- Parámetros económicos ---
P_EXP = 40.0        # €/MWh, precio de venta del excedente a la red
P_PEN = 80.0       # €/MWh, precio de compra de emergencia en el mercado spot
C_FIJO = 1000.0     # €/dia, coste fijo de operacion y mantenimiento
C_VAR_UNIT = 2.0    # €/MWh generado, desgaste de equipos

In [ ]:
#@title **Generador de escenarios**

def _generar_generacion_demanda(T):
    """
    A partir de un vector de escenarios climaticos T ya simulado, genera
    la generacion solar (Gs), eolica (Gw), hidraulica (Gh) y la demanda
    (D), cada una con la distribucion condicionada a su escenario.

    Esta funcion es el "nucleo" reutilizado tanto por el simulador del
    Bloque Basico (simulador) como por el del Bloque Avanzado (que la
    llama una vez por cada dia del anyo): solo cambia CUANTOS dias se
    generan de una vez, no la forma en que se simula cada dia dado T.
    """
    n = len(T)
    Gs = np.zeros(n); Gw = np.zeros(n); Gh = np.zeros(n); D = np.zeros(n)
    for tj, cfg in PARAMS_ESCENARIO.items():
        mask = (T == tj)
        n_j = int(mask.sum())
        if n_j == 0:
            continue
        ### Generador energia solar
        a, m, b = cfg["tri_s"]
        c_forma = (m - a) / (b - a)
        Gs[mask] = stats.triang.rvs(c=c_forma, loc=a, scale=(b - a), size=n_j)
        ### Generador energia eólica
        k_w, lam_w = cfg["weib_w"]
        Gw[mask] = stats.weibull_min.rvs(c=k_w, scale=lam_w, size=n_j)
        ### Generador energía hidráulica
        mu_h, sigma_h = cfg["ln_h"]
        Gh[mask] = stats.lognorm.rvs(s=sigma_h, scale=math.exp(mu_h), size=n_j)
        ### Demanda
        mu_d, sigma_d = cfg["norm_d"]
        D[mask] = stats.norm.rvs(loc=mu_d, scale=sigma_d, size=n_j)
    return Gs, Gw, Gh, D

In [ ]:
#@title **Simulador de días de funcionamiento**

def simulador(nsim):
    """Simula nsim días independientes de operación de la micro-red (Bloque Básico)."""
    ### Escenario
    T = np.random.choice(ESCENARIOS, size=nsim, p=PROBS_ESCENARIO)
    ### Generación y demanda
    Gs, Gw, Gh, D = _generar_generacion_demanda(T)
    G_total = Gs + Gw + Gh
    B = G_total - D
    ### Costes operativos
    C_var = C_VAR_UNIT * G_total
    Y = np.where(B >= 0, B * P_EXP, B * P_PEN) - C_FIJO - C_var
    ### Diccionario de resultados
    return pd.DataFrame({"Escenario": T, "Gen_Solar": Gs, "Gen_Eolica": Gw, "Gen_Hidraulica": Gh, "Demanda": D,
                          "Gen_Total": G_total, "Balance": B, "Coste": Y})

Podemos ver una simualción para 10000 días de funcionamiento del sistema con el código siguiente (fijamos la semilla para asegurar reproducibilidad):

In [ ]:
# Días simulados
nsim = 10000
# Semilla
np.random.seed(123)
# Generación de datos
datos_basico = simulador(nsim)
datos_basico.head(10)

,Escenario,Gen_Solar,Gen_Eolica,Gen_Hidraulica,Demanda,Gen_Total,Coste
0,2,15.432124,67.620200,9.525407,-26.266365,92.577732,-3286.464657
1,1,123.402989,4.775217,5.800277,-0.276616,133.978483,-1290.086209
2,1,131.657466,1.354756,9.731400,2.436284,142.743622,-1188.035902
3,2,13.541085,47.084995,12.539247,-32.074307,73.165326,-3712.275179
4,3,7.957602,22.439390,49.895738,-51.840087,80.292730,-5307.792408
5,2,16.793324,52.587155,11.952751,-17.931505,81.333230,-2597.186857
6,4,17.852837,11.967602,6.946488,-102.023671,36.766927,-9235.427515
7,2,0.870942,110.424838,14.115920,26.935446,125.411700,-173.405567
8,2,28.390074,67.012393,9.267976,-32.193992,104.670442,-3784.860243
9,1,130.756149,6.374471,6.460992,-2.046520,143.591612,-1450.904793


## **2.3. Verificar funcionamiento del algoritmo**

Antes de empezar a completar las tareas establecidas para analizar el comportamiento del sistema es necesario que verifiques las distribuciones asignadas en la descripción del proceso a partir de las 10000 simulaciones obtenidas.

Para dicha verificación puedes usar la función `gof_continuous` que te premite ajustar y estimar una distribución de tipo continuo  a un conjunto de datos. Para el análsiis de escenario climáticos basta con describir los resulttdos de dicha variable.

Te puedes ayudar tanto de resultados numéricos como de gráficos para verificar las distribuciones asumidas por la empresa y consideradas en el simulador. Presta especial atención a si la media y la varianza muestral de cada variable, calculadas por escenario, son coherentes con los parámetros teóricos de la Tabla de la sección 2.1.

## **2.4. El encargo de la Dirección**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección General, "Eco-Flow Renewables"
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Diagnóstico del riesgo financiero de la operación actual de la micro-red
>
> Antes de plantearnos cualquier inversión, necesitamos entender con números —no con impresiones— cuál es nuestra exposición real al riesgo con el modelo de negocio actual. Trabajad con la muestra de $nsim=10\,000$ días que habéis generado ya (o una mayor, si vuestro análisis lo requiere) y acompañad **cada estimación de su intervalo de confianza al 95%**: una cifra sin margen de error no nos sirve para tomar una decisión de este calado. Os hemos organizado la petición en tres objetivos.

### Objetivo 1. Cuantificar el riesgo económico de un día cualquiera

Necesitamos saber, en cifras concretas, qué podemos esperar en un día normal de operación y hasta qué punto se pueden torcer las cosas en un día malo.

* **O1.1.** Estimad el **beneficio diario esperado**, la **probabilidad de déficit** y el **Análisis de Riesgo Extremo (CVaR)** al 5% (el beneficio promedio en el 5% de los peores días).
* **O1.2.** Calculad el **VaR al 95%** (percentil 5 de $Y$) y comparadlo con el CVaR al 5% del punto anterior. Explicadnos qué información adicional nos aporta el CVaR frente al VaR.
* **O1.3.** Estimad el coeficiente de variación de $Y$ para cada escenario climático por separado. ¿En qué escenario es proporcionalmente más incierto nuestro resultado económico del día?

### Objetivo 2. Entender qué hay detrás de esa variabilidad

Antes de proponer ninguna solución, queremos saber qué parte de nuestro negocio explica realmente el riesgo.

* **O2.1.** Descomponed, por simulación, la varianza total de $Y$ en la parte atribuible al término de mercado ($B\cdot P$) y la parte atribuible al coste variable $C_{var}$. ¿Cuál domina la variabilidad de nuestro beneficio diario?
* **O2.2.** Diseñad un análisis de sensibilidad global que determine cuál de los parámetros del modelo (probabilidades de escenario, parámetros de las cuatro distribuciones condicionadas, o los precios $P_{exp}$/$P_{pen}$) explica mayor proporción de la varianza de $Y$: necesitamos saber dónde concentrar nuestros esfuerzos.

### Objetivo 3. Explorar palancas de mejora a corto plazo (sin invertir todavía en una batería)

Antes de comprometer capital en una batería de almacenamiento, queremos agotar las alternativas más baratas.

* **O3.1.** El precio de penalización $P_{pen}=80$€/MWh lo hemos supuesto constante, pero sabemos que el mercado spot es volátil. Sustituidlo por una variable aleatoria (por ejemplo, una Lognormal centrada en 80) y contadnos, por simulación, cómo cambiaría la distribución de $Y$ si tuviéramos que asumir esa volatilidad.
* **O3.2.** Planteadnos cómo evaluaríais, por simulación, si nos compensa más invertir en ampliar la capacidad solar instalada (lo que desplazaría hacia arriba la distribución de $G_s$) o firmar un contrato de compra de energía a plazo fijo (lo que eliminaría nuestra exposición a $P_{pen}$, a cambio de un precio pactado más alto que $P_{exp}$).
* **O3.3.** Necesitamos una cifra concreta: ¿cuánta capacidad solar adicional (en MWh de media) tendríamos que instalar para conseguir $E(Y)\ge 0$, manteniendo el resto de fuentes sin cambios?
* **O3.4.** Estamos valorando un programa de "respuesta de demanda": en los días de déficit severo, algunos clientes industriales aceptarían reducir su consumo un 10% a cambio de un descuento en su factura. Modelad esa reducción de $D$ condicionada a $B<0$ y estimad su efecto sobre $E(Y)$ y sobre $Pr(B<0)$.

# **<font color="brown">3. Gestión con bateria de almacenamiento</font>**

En el apartado anterior se trata cada día como un suceso aislado: todo el excedente de un día de superávit se vende de inmediato, y todo el déficit de un día malo se compra de inmediato en el mercado spot. En la práctica, "Eco-Flow Renewables" puede instalar una **batería de almacenamiento** que retiene parte del excedente de los días buenos para poder cubrir, total o parcialmente, el déficit de los días siguientes sin tener que acudir al mercado spot.

Esto introduce una diferencia fundamental respecto a todos los bloques anteriores de esta colección: el estado de la batería (cuánta energía tiene almacenada) **depende de lo ocurrido el día anterior**. Ya no podemos simular todos los días de forma independiente: hay que recorrerlos **secuencialmente**, actualizando el estado de carga día a día.

## **3.1. Modelo con bateria de almacenamiento**

Cada día $t=1,\dots,365$, la micro-red genera y consume energía exactamente igual que en el Bloque Básico, obteniendo el mismo balance neto $B_t=G_{total,t}-D_t$. La batería tiene una capacidad máxima $Cap=20$ MWh y un estado de carga $E_t$ (energía almacenada al empezar el día $t$, con $E_1=0$: empieza vacía). Además, como en toda batería real, no toda la energía almacenada se recupera: hay una **eficiencia de ida y vuelta** $\eta=0.90$ (se pierde un 10% de la energía en el ciclo de carga y descarga).

* **Si $B_t\ge 0$ (superávit):** primero se carga la batería hasta donde quepa, y el resto (si sobra) se vende a la red:

$$\text{carga}_t=\min(B_t,\ Cap-E_t), \qquad \text{excedente vendido}_t = B_t-\text{carga}_t$$

* **Si $B_t< 0$ (déficit):** primero se descarga la batería para cubrir el déficit (afectada por la eficiencia $\eta$), y lo que la batería no pueda cubrir se compra en el mercado spot:

$$\text{cobertura}_t=\min(-B_t,\ E_t\cdot\eta), \qquad \text{déficit residual}_t = -B_t-\text{cobertura}_t$$

(la energía realmente extraída del almacén es $\text{cobertura}_t/\eta\ge\text{cobertura}_t$, por las pérdidas).

El estado de carga se actualiza para el día siguiente ($E_{t+1} = E_t + \text{carga}_t - \text{cobertura}_t/\eta$, siempre acotado entre $0$ y $Cap$), y el beneficio neto del día se calcula igual que antes, pero sobre la parte efectivamente vendida o comprada:

$$Y_t = \text{excedente vendido}_t\cdot P_{exp} \ -\ \text{déficit residual}_t\cdot P_{pen} \ -\ C_{fijo} - C_{var,t}$$

| Parámetro | Símbolo | Valor |
| :--- | :--- | :--- |
| Capacidad de la batería | $Cap$ | $20$ MWh |
| Eficiencia de ida y vuelta | $\eta$ | $0.98$ |
| Estado de carga inicial | $E_1$ | $0$ MWh |
| Horizonte de simulación | — | $365$ días |

Para aislar el efecto puro de la batería, la emprsa considera generar **una única vez**, para cada uno de los 365 días de cada año simulado, la misma secuencia de escenario climático y de generación/demanda, y aplicamos sobre ella **ambas** políticas: "con batería" y "sin batería" (esta última es, día a día, idéntica al Bloque Básico). Así, cualquier diferencia en el beneficio anual se debe exclusivamente a la batería, y no a haber usado muestras climáticas distintas (**simulación pareada**).

> ⚠️ **Importante — no pierdas de vista la unidad de análisis.** Necesitamos recorrer los 365 días de cada año **en orden** porque la batería tiene memoria (el estado de carga de hoy depende del de ayer), pero eso es solo una necesidad *mecánica* de la simulación. La pregunta de interés — ¿cuánto vale un día cualquiera con batería frente a sin ella? — sigue siendo una pregunta sobre **el día**, exactamente igual que en el Bloque Básico. Por eso todo el análisis posterior (sección 3.4) se hace agrupando o promediando sobre los **días** de `datos_bateria` (36 500 filas si simulas 100 años), nunca sumando o promediando primero dentro de cada año: si agregases por año antes de estimar, dejarías de tener una estimación de "coste/beneficio de un día típico" y ya no podrías comparar el resultado con el del Bloque Básico.

## **3.2. Simulación del proceso**

En este punto describimos el pseudocódigo del algoritmo y se proporciona el código necesario para la simulación de un conjunto específico de días de funcionamiento, devolviendo un registro **por día**, exactamente con la misma filosofía que el simulador de la sección 2.2, para poder comparar directamente, día a día, el escenario "sin batería" y el escenario "con batería".

### **Pseudocódigo simulación del proceso**

1. Para cada uno de los $nsim$ años a simular, y cada uno de sus 365 días, simular (por el método de composición del Bloque Básico) el escenario climático $t$ y, condicionado a él, $G_s$, $G_w$, $G_h$ y $D$ — esta es la "base común". Calcular $G_{total}$ y el balance $B$ de cada día, igual que en el Bloque Básico.
2. Calcular, para cada día y de forma independiente, el **coste sin batería** aplicando directamente la regla de mercado del Bloque Básico.
3. Recorrer los 365 días **en orden** manteniendo el estado de carga $E_t$, y calcular para cada día, según corresponda:
    * si $B_t\ge 0$: la energía cargada en la batería y el excedente vendido a la red;
    * si $B_t< 0$: la energía cubierta por la batería (cobertura) y el déficit residual comprado en el mercado spot;
    * en ambos casos, el nuevo estado de carga $E_{t+1}$ y el **coste con batería** de ese día.
4. Construir, con toda esta información, un único registro (una fila) por cada día y cada año simulado, con columnas análogas a las del Bloque Básico (escenario, generación, demanda, balance) más las columnas propias de la batería (estado de carga, carga del día, cobertura del día, excedente vendido, déficit residual) y **ambos** costes diarios (con y sin batería), para poder comparar directamente ambos modelos.


### **Código simulación del proceso**

A continuación se presentan las funciones necesarias para tener en cuenta la presencia de la batería, reutilizando en todo momento el simulador emparejado (Bloque Básico) para generar la base común de datos climáticos.

In [ ]:
# @title **Parámetros nuevos con la inclusión de batería de respaldo**
N_DIAS = 365
BAT_CAP = 20.0          # MWh, capacidad de la bateria
EFICIENCIA_BAT = 0.95   # eficiencia de ida y vuelta (round-trip)

In [ ]:
#@title **Generador de la base común**
def _generar_base_anual(nsim):
    """
    Genera, para nsim anyos independientes, el escenario climatico T y las
    variables de generacion/demanda de cada uno de los N_DIAS dias. Usa la
    MISMA funcion _generar_generacion_demanda del Bloque Basico, aplicada
    de una sola vez sobre un vector de nsim*N_DIAS dias "aplanado".

    Devuelve arrays de forma (nsim, N_DIAS): esta es la "base comun" que
    reutilizaran, sin cambios, los dos escenarios (con y sin bateria).
    """
    T = np.random.choice(ESCENARIOS, size=(nsim, N_DIAS), p=PROBS_ESCENARIO)
    Gs, Gw, Gh, D = _generar_generacion_demanda(T.ravel())
    forma = (nsim, N_DIAS)
    return T, Gs.reshape(forma), Gw.reshape(forma), Gh.reshape(forma), D.reshape(forma)

In [ ]:
def simular_anyo_con_bateria(base):
    """
    Recorre los N_DIAS dias EN ORDEN (bucle sobre los dias, vectorizado
    sobre los nsim anyos simultaneamente) manteniendo el estado de carga
    E de la bateria.

    A diferencia de versiones anteriores, esta funcion devuelve un
    DataFrame "largo" con UNA FILA POR CADA (anyo, dia) -- igual que el
    simulador() del Bloque Basico devuelve una fila por dia -- para que
    ambos modelos se puedan comparar directamente dia a dia. Incluye:

    - Escenario, Gen_Solar, Gen_Eolica, Gen_Hidraulica, Gen_Total,
      Demanda, Balance: identicas magnitudes que en el Bloque Basico.
    - Carga_bateria_inicio / Carga_bateria_fin: estado de carga de la
      bateria al EMPEZAR y al TERMINAR ese dia.
    - Carga_dia: energia efectivamente cargada en la bateria ese dia
      (>0 solo en dias de superavit).
    - Cobertura_dia: energia efectivamente cubierta por la bateria ese
      dia (>0 solo en dias de deficit).
    - Excedente_vendido / Deficit_residual: lo que, tras pasar por la
      bateria, se acaba vendiendo a la red o comprando en el spot.
    - Coste_sin_bateria / Coste_con_bateria: el coste de ESE MISMO dia
      bajo las dos politicas, calculadas sobre la misma base comun
      (numeros aleatorios comunes), listas para comparar o restar.
    """
    T, Gs, Gw, Gh, D = base
    nsim, ndias = Gs.shape
    G_total = Gs + Gw + Gh
    B = G_total - D
    C_var = C_VAR_UNIT * G_total

    # --- Coste "sin bateria", dia a dia e independiente (regla del Bloque Basico) ---
    Coste_sin_bateria = np.where(B >= 0, B * P_EXP, B * P_PEN) - C_FIJO - C_var

    # --- Recorrido secuencial dia a dia para la politica "con bateria" ---
    E = np.zeros(nsim)   # estado de carga inicial: bateria vacia (E_1 = 0)
    Carga_bateria_inicio = np.empty((nsim, ndias))
    Carga_bateria_fin = np.empty((nsim, ndias))
    Carga_dia = np.empty((nsim, ndias))
    Cobertura_dia = np.empty((nsim, ndias))
    Excedente_vendido = np.empty((nsim, ndias))
    Deficit_residual = np.empty((nsim, ndias))
    Coste_con_bateria = np.empty((nsim, ndias))

    for t in range(ndias):
        Bt = B[:, t]
        superavit = Bt >= 0
        Carga_bateria_inicio[:, t] = E

        # Dias de superavit: cargar bateria hasta donde quepa, vender el resto
        carga = np.where(superavit, np.minimum(Bt, BAT_CAP - E), 0.0)
        excedente_vendido = np.where(superavit, Bt - carga, 0.0)

        # Dias de deficit: descargar bateria (con perdidas por eficiencia), comprar el resto
        descarga_util_max = E * EFICIENCIA_BAT
        cobertura = np.where(~superavit, np.minimum(-Bt, descarga_util_max), 0.0)
        consumo_almacen = cobertura / EFICIENCIA_BAT
        deficit_residual = np.where(~superavit, -Bt - cobertura, 0.0)

        # Actualizacion del estado de carga para el dia siguiente
        E = np.clip(E + carga - consumo_almacen, 0.0, BAT_CAP)

        ingreso = excedente_vendido * P_EXP
        coste_deficit = deficit_residual * P_PEN
        Coste_con_bateria[:, t] = ingreso - coste_deficit - C_FIJO - C_var[:, t]

        Carga_bateria_fin[:, t] = E
        Carga_dia[:, t] = carga
        Cobertura_dia[:, t] = cobertura
        Excedente_vendido[:, t] = excedente_vendido
        Deficit_residual[:, t] = deficit_residual

    # --- Montamos el registro largo: una fila por cada (anyo, dia) ---
    Anyo_idx = np.repeat(np.arange(1, nsim + 1), ndias)
    Dia_idx = np.tile(np.arange(1, ndias + 1), nsim)

    datos = pd.DataFrame({
        "Anyo": Anyo_idx,
        "Dia": Dia_idx,
        "Escenario": T.ravel(),
        "Gen_Solar": Gs.ravel(),
        "Gen_Eolica": Gw.ravel(),
        "Gen_Hidraulica": Gh.ravel(),
        "Gen_Total": G_total.ravel(),
        "Demanda": D.ravel(),
        "Balance": B.ravel(),
        "Carga_bateria_inicio": Carga_bateria_inicio.ravel(),
        "Carga_dia": Carga_dia.ravel(),
        "Cobertura_dia": Cobertura_dia.ravel(),
        "Carga_bateria_fin": Carga_bateria_fin.ravel(),
        "Excedente_vendido": Excedente_vendido.ravel(),
        "Deficit_residual": Deficit_residual.ravel(),
        "Coste_sin_bateria": Coste_sin_bateria.ravel(),
        "Coste_con_bateria": Coste_con_bateria.ravel(),
    })
    return datos

Podemos ver una simulación para el funcionamiento del sistema durante 15 años (fijamos la semilla para asegurar reproducibilidad). Simulamos años completos porque la batería obliga a recorrer los días en orden, pero el resultado sigue siendo un registro con **una fila por día**, directamente comparable, día a día, con el del Bloque Básico — los "años" no son la unidad de análisis, son solo la forma en que hemos organizado la generación de los datos:

In [ ]:
# Anyos simulados
nsim_bateria = 15
# Semilla
np.random.seed(123)
# Generacion de la base comun y simulacion con bateria (incluye el coste sin bateria, para comparar)
base_anual = _generar_base_anual(nsim_bateria)
datos_bateria = simular_anyo_con_bateria(base_anual)
datos_bateria.head(10)

## **3.3. Verificar funcionamiento del algoritmo**

Antes de pasar a las tareas, comprueba que tu simulador de la batería es correcto. Algunas verificaciones que deberías poder superar con el registro `datos_bateria` obtenido:

* El estado de carga (`Carga_bateria_inicio` y `Carga_bateria_fin`) debe estar siempre entre $0$ y $Cap=20$ MWh, para cualquier día y cualquier año.
* Para cada día, debe cumplirse la identidad de balance de energía de la batería: `Carga_bateria_fin = Carga_bateria_inicio + Carga_dia - Cobertura_dia / EFICIENCIA_BAT` (recuerda acotarla después entre $0$ y $Cap$).
* En los días en que la batería no interviene (`Carga_dia = 0` y `Cobertura_dia = 0`), el coste con batería debe coincidir exactamente con el coste sin batería: la batería, cuando no actúa, no puede cambiar el resultado del día.
* El primer día de cada año (`Dia = 1`), `Carga_bateria_inicio` debe valer $0$ para todos los años, puesto que $E_1=0$ es una hipótesis del modelo.
* Como comprobación adicional, agrupa `datos_bateria` por escenario climático y compara la generación y demanda medias con las que obtuviste en la sección 2.3 para el Bloque Básico: deben ser coherentes, ya que ambos bloques comparten la misma función `_generar_generacion_demanda()`.

Si alguna de estas verificaciones falla, revisa tu implementación antes de continuar: un error aquí invalidaría todas las estimaciones posteriores sobre la rentabilidad de la batería.

## **3.4. El encargo de la Dirección (fase 2): ¿compensa instalar la batería?**

> **MEMORÁNDUM INTERNO**
>
> **De:** Dirección General, "Eco-Flow Renewables"
> **Para:** Equipo de Ciencia de Datos
> **Asunto:** Evaluación de la inversión en la batería de almacenamiento de 20 MWh
>
> Gracias por el diagnóstico de la fase anterior. Ahora necesitamos que evaluéis si la batería de 20 MWh que nos propone el proveedor es una inversión que compensa. Trabajad siempre directamente sobre las filas de `datos_bateria` (una fila por día), exactamente igual que hicisteis con `datos_basico`: no agrupéis ni suméis antes por año — la columna `Anyo` solo sirve aquí para identificar la secuencia a la que pertenece cada día. Como siempre, acompañad cada estimación de su intervalo de confianza al 95%. Os hemos organizado la petición en cuatro objetivos.

### Objetivo 4. Comparar, día a día, el funcionamiento con y sin batería

* **O4.1.** Estimad el **coste/beneficio diario esperado** con y sin batería, y la **ganancia media diaria pareada** por instalarla. Aprovechad que el diseño es pareado (misma base climática en ambas columnas de coste) para explicarnos por qué el intervalo de confianza de la ganancia sale mucho más estrecho que si hubierais simulado ambos escenarios de forma independiente.
* **O4.2.** Calculad la **probabilidad de que un día cualquiera nos obligue a acudir al mercado spot**, con y sin batería, y comparadla con la $Pr(B<0)$ que estimasteis en O1.1. ¿En qué proporción de los días de déficit consigue la batería evitarnos por completo la compra en el mercado spot?
* **O4.3.** Complementad el punto anterior con el **CVaR al 5%** del coste diario, con y sin batería. ¿Nos protege más la batería en el día medio, o en el día extremo?

### Objetivo 5. Saber de qué depende el valor de la batería

* **O5.1.** Estudiad cómo cambiaría la ganancia diaria esperada de la batería (O4.1) si su capacidad se duplicase a 40 MWh o se redujese a la mitad (10 MWh). ¿La relación entre capacidad y ganancia es lineal, o presenta rendimientos decrecientes?
* **O5.2.** Cuantificad el efecto de la eficiencia de ida y vuelta $\eta$ sobre esa ganancia diaria: repetid el análisis con $\eta=1.0$ (batería ideal) y con $\eta=0.90$ (batería más deficiente), y comparad con nuestro valor de referencia $\eta=0.98$.
* **O5.3.** Descomponed la ganancia diaria esperada de la batería según el escenario climático de cada día. ¿En qué escenario climático nos resulta más valiosa la batería? Relacionadlo con la frecuencia y magnitud del déficit en cada escenario (O1.1 y O1.3).

### Objetivo 6. Dimensionar la inversión y comprobar su robustez frente a un mal año

* **O6.1.** Planteadnos el problema de **dimensionamiento óptimo**: si instalar la batería cuesta una cantidad fija anualizada (amortización) que crece con su capacidad, describid cómo usaríais el simulador para encontrar la capacidad $Cap^{*}$ que maximiza nuestro beneficio neto anual esperado. Para pasar de la ganancia diaria (O4.1 o O5.1) a una cifra anual, multiplicadla por 365 — es una simple conversión de unidades, no hace falta una nueva simulación agregada por año.
* **O6.2.** Nos preocupan especialmente las rachas prolongadas de mal tiempo. Diseñad un experimento que fuerce una racha de 20 días consecutivos en escenario Anticiclónico dentro de un año, y seguid día a día cuánto se vacía la batería durante la racha y cuánto tarda en recuperarse después.
* **O6.3.** Calculad el **Loss of Load Expectation (LOLE)**: el número medio de días al año en los que nuestra demanda no queda cubierta, con y sin batería. Expresadlo como $LOLE \approx 365 \times Pr(\text{Deficit\_residual}>0)$, reutilizando las probabilidades diarias de O4.2, en lugar de contarlo año a año. ¿En cuántos días al año nos libra la batería de acudir al mercado spot?

### Objetivo 7. Vuestra recomendación

* **O7.1.** A partir de los resultados de los Objetivos 4 a 6, redactadnos la recomendación de inversión sobre la batería de 20 MWh: ¿la instalaríais? ¿bajo qué condiciones (de capacidad, de coste de amortización) cambiaría vuestra recomendación? Justificadla con números concretos y sus intervalos de confianza, no con impresiones cualitativas.

## Objetivo 8. Ampliación optativa

A diferencia de los Objetivos 4-7 (que trabajan siempre sobre el resultado de un día individual, para poder compararlo con el Bloque Básico), varias de estas cuestiones preguntan deliberadamente por el **beneficio agregado de un año completo** ($Y_{anual}=\sum_{t=1}^{365} Y_t$) como objeto de estudio en sí mismo. Tened claro en cada cuestión cuál de las dos unidades de análisis (el día o el año) se os pide, y no las mezcléis dentro de una misma respuesta.

* **O8.1.** Diseñad una prueba de hipótesis, o un procedimiento por simulación pareada, para contrastar si la ganancia media de la batería es significativamente distinta de cero al nivel de confianza del 95%.
* **O8.2.** El estado inicial de la batería se ha fijado en $E_1=0$ (vacía). Evaluad, por simulación, si el beneficio anual esperado depende de forma apreciable de si el año empieza con la batería vacía, llena, o a mitad de carga.
* **O8.3.** Diseñad una **estrategia de gestión activa** de la batería: en lugar de la regla reactiva actual (cargar solo si hay superávit, descargar solo si hay déficit), permitid que la batería se descargue deliberadamente para vender energía cuando el precio de exportación suba (si el modelo incorporase un precio de venta variable en el tiempo). Describid cómo cambiaría la función `simular_anyo_con_bateria`.
* **O8.4.** Planteadnos cómo incorporaríais el **valor de la información meteorológica**: si dispusiéramos de una predicción fiable del escenario climático de mañana, podríamos decidir hoy no vender todo el excedente y reservarlo en la batería. Describid, a alto nivel, cómo evaluaríais por simulación el valor económico de esa información.
* **O8.5.** Un regulador nos ofrece una tarifa especial de "servicios de red" que paga por tener capacidad de almacenamiento disponible, independientemente de si la usamos o no. Incorporad este ingreso adicional al modelo y estimad, por simulación, a partir de qué tarifa mínima la instalación de la batería sería rentable por sí sola.
* **O8.6.** Diseñad una comparación honesta y completa (coste de capital, ganancia operativa anual, riesgo residual) entre instalar la batería de 20 MWh frente a ampliar directamente la capacidad de generación solar en una cantidad equivalente en coste. ¿Qué información adicional, no disponible en este simulador, necesitaríais para completar esa comparación?